# FINAL: DMA workable build (LEN=512 verified idle+refresh) + BPM

DMA 已验证 256/512/1024 idle且刷新(你的扫描结果)。采用 **LEN=512**。
G1: 每帧连采 B_PER 个 512 块拼接成完整音频波形, 滚动显示。
G2: BPM 读寄存器 0x40/0x50/0x60, 与DMA无关。
先清Output/断电上电, 放鼓点跑。


In [ ]:
import time, struct, numpy as np
from pynq import Overlay, allocate
import smbus2
from smbus2 import i2c_msg

GAIN   = 0x20       # 音量增益: 过大(0x7F)阈值自锁BPM=0
LEN    = 512        # 已验证: 该长度DMA能idle且刷新(核心!)
B_PER  = 30         # 每帧拼接块数 -> 波形长度 = LEN*B_PER 采样
NF     = 40         # BPM曲线帧数
PER    = 0.12
Fs     = 48800.0
WIN_S  = 2.0        # 波形横坐标窗口秒数
PASS = lambda m: print('[PASS] ' + m)
FAIL = lambda m: print('[FAIL] ' + m)
print('[0] LEN=%d B_PER=%d win=%.1fs GAIN=0x%02X' % (LEN, B_PER, WIN_S, GAIN))


In [ ]:
ol = Overlay('design_1.bit')
seg, bpm, dma, adau = ol.segment_stream_0, ol.bpm_detect_0, ol.axi_dma_0, ol.adau1761_0
print('[1] overlay loaded')


In [ ]:
bus = smbus2.SMBus(0); CAD = 0x3B
def wr16(r, v):
    bus.i2c_rdwr(i2c_msg.write(CAD, [(r>>8)&0xFF, r&0xFF, v])); time.sleep(0.002)
wr16(0x4000,0x00); wr16(0x4000,0x03); wr16(0x40F9,0x7F); wr16(0x40FA,0x01)
wr16(0x4015,0x00); wr16(0x4016,0x00); wr16(0x4017,0x00)
for r, v in [(0x400A,0x11),(0x400B,0x10),(0x400C,0x11),(0x400D,0x10),
             (0x400E,GAIN),(0x400F,GAIN),(0x4010,0x01),(0x4026,0x33)]:
    wr16(r, v)
print('[2] codec configured')


In [ ]:
buf = allocate(shape=(LEN,), dtype=np.uint32)
dma.recvchannel.start()

def grab_block():
    """采一个 512 块 (已验证 idle+刷新)"""
    try: dma.recvchannel.transfer(buf)
    except Exception: pass
    buf.flush()
    seg.mmio.write(0x00, LEN)
    time.sleep(0.05)
    buf.invalidate()
    return np.asarray(buf).copy()

def grab_wave():
    """拼 B_PER 个块成一条完整波形 (每块取左声道)"""
    ch = []
    for _ in range(B_PER):
        ra = grab_block()
        x = (ra[::2] ^ 0x800000).astype(np.int32) - 0x800000
        ch.append(x)
    return np.concatenate(ch)

def read_bpm():
    try: return struct.unpack('<f', int(bpm.mmio.read(0x40)).to_bytes(4,'little'))[0]
    except Exception: return 0.0
print('[3] DMA(512) + BPM ready')


## 现在放鼓点/节拍器, 实时采集完整波形 + BPM 曲线


In [ ]:
from collections import deque
import matplotlib.pyplot as plt
plt.rcParams['axes.unicode_minus'] = False

SAM = LEN * B_PER // 2           # 每帧完整波形采样数(左声道)
wave_cat = []                    # 每帧原始波形(左声道)累积
hist_bpm, hist_dbgn, hist_sw = [], [], []
abs_time = 0.0
prev = None
same = 0

print('>> PLAY DRUM / METRONOME now...')
for i in range(NF):
    x = grab_wave()
    abs_time += x.shape[0] / Fs
    wave_cat.append(x)
    sv = int(x.max() - x.min())
    dbg = bpm.mmio.read(0x50); env = bpm.mmio.read(0x60); b2 = read_bpm()
    hist_bpm.append(b2); hist_dbgn.append(dbg); hist_sw.append(sv)
    if prev is not None and np.array_equal(prev, x): same += 1
    prev = x.copy()
    print('  [%d] swing=%7d BPM=%6.1f peaks=%d env=0x%08X' % (i, sv, b2, dbg, env))
    time.sleep(PER)

# ---- 一次性静态绘图: 保证 Jupyter inline 下波形一定可见 ----
allw = np.concatenate(wave_cat)
tw = np.arange(len(allw)) / Fs
dur = len(allw) / Fs
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(13, 9))
ax1.plot(tw, allw, lw=0.4, color='C0')
ax1.set_xlabel('Time (s)'); ax1.set_ylabel('Amp'); ax1.grid(alpha=0.3)
ax1.set_xlim(0, dur)
ax1.set_title('G1 DMA Audio Waveform  (LEN=%d, %.1fs captured) swing_max=%d'
              % (LEN, dur, max(hist_sw) if hist_sw else 0))
ax2.plot(np.arange(len(hist_bpm)), hist_bpm, 'o-', color='C3')
ax2.set_xlabel('frame #'); ax2.set_ylabel('BPM'); ax2.grid(alpha=0.3)
ax2.set_ylim(0, 240)
ax2.set_title('G2 BPM Curve  final BPM=%.1f' % (hist_bpm[-1] if hist_bpm else 0))
fig.tight_layout()
plt.show()

good = sum(1 for b in hist_bpm if b > 0)
if good >= 2: PASS('BPM>0 on %d/%d frames' % (good, NF))
else:         FAIL('BPM ~0 -> lower GAIN to 0x10/0x08')
if same >= NF - 2: FAIL('DATA NOT REFRESHING (same=%d)' % same)
else:              PASS('DATA REFRESHING (same=%d) -> DMA works!' % same)
print('done, total=%.1fs' % dur)
